In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig
from tqdm import tqdm
import numpy as np
import sqlite3

# ----------------------
#  Model Components
# ----------------------
class ChannelAttention(nn.Module):
    """Channel-wise attention module using both average and max pooling"""
    def __init__(self, in_channels: int, reduction: int = 4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction),
            nn.GELU(),
            nn.Linear(in_channels // reduction, in_channels),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg_out = self.fc(self.avg_pool(x).squeeze(-1))
        max_out = self.fc(self.max_pool(x).squeeze(-1))
        return x * (avg_out + max_out).unsqueeze(-1)

class CrossScaleAttention(nn.Module):
    """Cross-scale attention for feature fusion between different resolutions."""
    def __init__(self, curr_channels: int, prev_channels: int):
        super().__init__()
        self.query_proj = nn.Conv1d(curr_channels, curr_channels, 1)
        self.key_proj = nn.Conv1d(prev_channels, curr_channels, 1)
        self.value_proj = nn.Conv1d(prev_channels, curr_channels, 1)

    def forward(self, curr_features: torch.Tensor, prev_features: torch.Tensor) -> torch.Tensor:
        # Align feature dimensions
        if prev_features.shape[-1] != curr_features.shape[-1]:
            prev_features = F.interpolate(
                prev_features, 
                size=curr_features.shape[-1],
                mode='linear',
                align_corners=False
            )
            
        # Compute attention weights
        Q = self.query_proj(curr_features)
        K = self.key_proj(prev_features)
        V = self.value_proj(prev_features)
        
        attn_scores = torch.einsum('bcl,bcl->bl', Q, K).unsqueeze(1)
        attn_weights = torch.sigmoid(attn_scores)
        
        return curr_features + attn_weights * V

class ResidualBlock(nn.Module):
    """Residual block with optional channel attention"""
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1, use_attn: bool = True):
        super().__init__()
        
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, in_ch, 3, stride, 1, groups=in_ch),
            nn.BatchNorm1d(in_ch),
            nn.GELU(),
            nn.Conv1d(in_ch, out_ch, 1),
            nn.BatchNorm1d(out_ch),
            nn.Dropout(0.2)
        )

        self.shortcut = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, stride),
            nn.BatchNorm1d(out_ch)
        ) if (stride != 1 or in_ch != out_ch) else nn.Identity()

        self.attn = ChannelAttention(out_ch) if use_attn else nn.Identity()
        self.activation = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.shortcut(x)
        x_conv = self.conv(x)
        x_att = self.attn(x_conv)
        return self.activation(x_att + residual)

class PosTransformer(nn.Module):
    """Position-aware Transformer block for Conv1d feature"""
    def __init__(self, dim: int, depth: int = 2, heads: int = 4):
        super().__init__()
        self.pos_enc = nn.Parameter(torch.randn(1, 1, dim))
        self.norm = nn.LayerNorm(dim)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=dim,
                nhead=heads,
                dim_feedforward=dim*4,
                dropout=0.2,
                activation='gelu'
            ) for _ in range(depth)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, L = x.shape
        x = x + self.pos_enc[:, :, :L]
        x = self.norm(x.permute(0, 2, 1))  # [B, C, L] --> [B, L, C]
        for layer in self.layers:
            x = layer(x)
        return x.permute(0, 2, 1)  # [B, L, C] --> [B, C, L]

# ---------------------
#  Main Model
# ---------------------
class SpectralPredictor(nn.Module):
    def __init__(self, embed_dim: int = 1154, meta_dim: int = 6, spec_dim: int = 14000):
        super().__init__()

        self.register_buffer('proton_mass', torch.tensor(1.007276))
        self.register_buffer('bin_centers', torch.arange(0, 1400, 0.1)[:spec_dim])

        self.fusion = nn.Sequential(
            nn.Conv1d(embed_dim + meta_dim, 1024, 3, padding=1),
            nn.BatchNorm1d(1024),
            ChannelAttention(1024),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.stage1 = self._build_stage(1024, 1024, 5, True)
        self.downsample = ResidualBlock(1024, 2048, stride=2)
        self.cross_attn = CrossScaleAttention(2048, 1024)
        self.stage2 = self._build_stage(2048, 2048, 5, True)

        self.pred_head = nn.Sequential(
            nn.Linear(2048 * 4, 4096),
            nn.LayerNorm(4096),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(4096, spec_dim),
            nn.Softplus()
        )
        
        self.length_head = nn.Sequential(
            nn.Linear(2048 * 4, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(512, 9)
        )
        
        self.charge_head = nn.Sequential(
            nn.Linear(2048 * 4, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(512, 3)
        )

    def _build_stage(self, in_ch, out_ch, blocks, use_transformer):
        layers = [ResidualBlock(in_ch, out_ch) for _ in range(blocks)]
        if use_transformer:
            layers.append(PosTransformer(out_ch))
        return nn.Sequential(*layers)

    def forward(self, embed: torch.Tensor, meta: torch.Tensor, precursor_mz: torch.Tensor) -> torch.Tensor:
        embed = embed.permute(0, 2, 1)     # [B, C, L] --> [B, L, C]
        meta_expanded = meta.unsqueeze(-1).expand(-1, -1, embed.shape[-1])
        x = self.fusion(torch.cat([embed, meta_expanded], dim=1))

        s1 = self.stage1(x)
        x_down = self.downsample(s1)
        x_att = self.cross_attn(x_down, s1)
        s2 = self.stage2(x_att)

        avg_pool = F.adaptive_avg_pool1d(s2, 2).flatten(1)
        max_pool = F.adaptive_max_pool1d(s2, 2).flatten(1)
        pool_features = torch.cat([avg_pool, max_pool], dim=1)

        charge = torch.argmax(meta[:, :3], dim=1) + 1
        mass = precursor_mz * charge - (charge - 1) * self.proton_mass + 0.1
        mask = (self.bin_centers.unsqueeze(0) <= mass.unsqueeze(-1)).float()
        
        raw_pred = self.pred_head(pool_features)
        length_logits = self.length_head(pool_features)
        charge_logits = self.charge_head(pool_features)
        return raw_pred, mask, length_logits, charge_logits

# --------------------------
# Prediction Function
# --------------------------
MODEL_PATH = r"your path\OligoPioneer.pt"

def predict_and_export(input_db_path: str, output_msp_path: str, chunk_size=256):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Load esmc models
    model_esmc = ESMC.from_pretrained("esmc_600m").to(device)
    model_esmc.eval()
    
    model = SpectralPredictor().to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
    model.eval()

    mass = {
        'A': 71.0371, 'C': 103.0091, 'D': 115.0269, 'E': 129.0426, 'F': 147.0684,
        'G': 57.0214, 'H': 137.0589, 'I': 113.0840, 'K': 128.0949, 'L': 113.0840,
        'M': 131.0404, 'N': 114.0429, 'P': 97.0527, 'Q': 128.0585, 'R': 156.1011,
        'S': 87.0320, 'T': 101.0476, 'V': 99.0684, 'W': 186.0793, 'Y': 163.0633
    }
    mass_scale = 200.0

    # Read data from the SQLite database
    table_names = ["peptide_charge1", "peptide_charge2"]
    with open(output_msp_path, 'w') as f:
        for table_name in table_names:
            print(f"🔍 Processing table: {table_name}")
            
            conn = sqlite3.connect(input_db_path)
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            
            # Check whether the required columns exist
            required_columns = ['Sequence', 'Charge', 'CE', 'PrecursorMz']
            missing_columns = [col for col in required_columns if col not in df.columns]
            if missing_columns:
                print(f"⚠️ Table {table_name} is missing required columns: {', '.join(missing_columns)}，skipping")
                continue
                
            has_comment = 'Comment' in df.columns
            
           # Process the current table in chunks
            for chunk_start in tqdm(range(0, len(df), chunk_size), desc=f"Processing {table_name}"):
                chunk_embeddings = []
                meta_features = None
                pred_spectra = None
                
                try:
                    chunk_end = min(chunk_start + chunk_size, len(df))
                    chunk_df = df.iloc[chunk_start:chunk_end].copy()
                    
                    # Calculate NCE values
                    charge_factors = {1: 1.0, 2: 0.9, 3: 0.85}
                    chunk_df['Charge'] = chunk_df['Charge'].astype(int)
                    chunk_df['NCE'] = (chunk_df['CE'] * 500) / (chunk_df['PrecursorMz'] * chunk_df['Charge'].map(charge_factors))
                    
                    # ========== Generate embeddings ==========
                    for _, row in chunk_df.iterrows():
                        seq = row['Sequence']
                        seq_len = len(seq)
                        protein = ESMProtein(sequence=seq)
                        token = model_esmc.encode(protein)
                        with torch.no_grad():
                            logits = model_esmc.logits(token, LogitsConfig(sequence=True, return_embeddings=True))
                        emb = logits.embeddings.squeeze(0)

                        aa_mass = torch.zeros(12, device=device)
                        peptide_bond_count = torch.zeros(12, device=device)
                        
                        for j, aa in enumerate(seq):
                            pos_in_embedding = j + 1  
                            if pos_in_embedding < 12:  
                                aa_mass[pos_in_embedding] = mass.get(aa, 0.0) / mass_scale

                        for j in range(12):
                            if 1 <= j <= seq_len:
                                peptide_bond_count[j] = (j - 1) / 100
                            
                        pad_length = 12 - emb.shape[0]
                        if pad_length > 0:
                            padding = torch.zeros((pad_length, emb.shape[1]), device=device)
                            padded_emb = torch.cat([emb, padding], dim=0)
                        else:
                            padded_emb = emb[:12, :]

                        enhanced_emb = torch.cat([padded_emb, aa_mass.unsqueeze(1), peptide_bond_count.unsqueeze(1)], dim=1)
                        chunk_embeddings.append(enhanced_emb.cpu())
                        
                        del protein, token, logits, emb, aa_mass, peptide_bond_count, enhanced_emb
                        torch.cuda.empty_cache()

                    # ========== Process metadata ==========
                    charge_indices = chunk_df['Charge'].map({1:0, 2:1, 3:2}).astype(int).tolist()
                    charge_onehot = F.one_hot(torch.tensor(charge_indices), num_classes=3).float()
                    precursormz = chunk_df['PrecursorMz'].values / 800
                    ce = chunk_df['CE'].values / 100
                    nce = chunk_df['NCE'].values / 100
                    
                    meta_features = torch.cat([
                        charge_onehot,
                        torch.tensor(precursormz, dtype=torch.float32).unsqueeze(1),
                        torch.tensor(ce, dtype=torch.float32).unsqueeze(1),
                        torch.tensor(nce, dtype=torch.float32).unsqueeze(1)
                    ], dim=1).to(device)

                    # ========== Prediction ==========
                    with torch.no_grad():
                        embeddings_tensor = torch.stack(chunk_embeddings).to(device)
                        precursor_mz = torch.tensor(chunk_df['PrecursorMz'].values, dtype=torch.float32).to(device)
                        
                        raw_pred, mask, _, _ = model(embeddings_tensor, meta_features, precursor_mz)
                        pred_spectra = (raw_pred * mask) ** 2
                        pred_spectra = pred_spectra.cpu().numpy()
                        
                        del embeddings_tensor, precursor_mz, raw_pred, mask
                        torch.cuda.empty_cache()

                    # ================= Write results =================
                    charge_to_precursor = {1: "[M+H]+", 2: "[M+2H]2+", 3: "[M+3H]3+"}
                
                    for idx, (_, row) in enumerate(chunk_df.iterrows()):
                        spectrum = pred_spectra[idx]
                    
                        max_intensity = np.max(spectrum)
                        spec_norm = (spectrum / max_intensity) * 1000.0
                        spec_filtered = np.where(spec_norm >= 10.0, spec_norm, 0.0)
                    
                        entry = [
                            f"Name: {row['Sequence']}",
                            f"PrecursorType: {charge_to_precursor.get(row['Charge'])}",
                            f"PrecursorMz: {row['PrecursorMz']:.4f}",
                            f"CollisionEnergy: {row['CE']:.2f}"
                        ]

                        if has_comment:
                            entry.append(f"Comment: {row['Comment']}")
                    
                        entry.append("Num peaks: 0")

                        peaks = []
                        for bin_idx in range(len(spec_filtered)):
                            intensity = spec_filtered[bin_idx]
                            if intensity > 0:
                                mz = bin_idx * 0.1
                                peaks.append(f"{mz:.4f} {intensity:.4f}")
                    
                        entry[-1] = f"Num peaks: {len(peaks)}"
                        entry.extend(peaks)
                        f.write("\n".join(entry) + "\n\n")

                finally:
                    vars_to_clear = ['chunk_embeddings', 'meta_features', 'pred_spectra']
                    for var in vars_to_clear:
                        if var in locals():
                            del locals()[var]
                    torch.cuda.empty_cache()

    print(f"✅ Prediction complete! Results saved to {output_msp_path}")

if __name__ == "__main__":
    predict_and_export(
        input_db_path=r"your path\XXX.db",
        output_msp_path=r"your path\XXX.msp",
        chunk_size=1024
    )